Chapter 7: Finetuning To Follow Instructions

1) Building an LLM
-1) data preparation and sampling 
-2) Attention mechenism
-3) LLM architecture

2) Foundation model

In [ ]:
from importlib.metadata import version, PackageNotFoundError

# Used to check installed package versions and handle missing packages


pkgs = ["numpy", "matplotlib", "torch", "tensorflow", "tqdm", "tiktoken"]

for p in pkgs:
    try:
        print(f"{p} version: {version(p)}")
    except PackageNotFoundError:
        print(f"{p} is NOT installed")

numpy version: 2.3.3
matplotlib version: 3.10.6
torch version: 2.9.1+cpu
tensorflow is NOT installed
tqdm version: 4.67.3


In [5]:
pip install -U tensorflow

   ━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.3/620.7 MB 2.8 MB/s eta 0:03:22
ERROR: Operation cancelled by user
^C
Note: you may need to restart the kernel to use updated packages.


7.1 Introduction to instruction finetuning

In chapter 5, we saw that pretraining an LLM involves a training procedure where it learns to generate one word at a time
Hence, a pretrained LLM is good at text completion, but it is not good at following instructions
In this chapter, we teach the LLM to follow instructions better

----------------------------------------------------------------------------------------------------------

Instruction:

Summarize the following text in one sentence:

“Artificial Intelligence enables machines to learn from data, identify patterns, and make decisions with minimal human intervention.”

Desired response:

Artificial Intelligence allows machines to learn from data and make decisions automatically.

-------------------------------------------------------------------------------------------------------------------------------------

7.2 Preparing a dataset for supervised instruction finetuning

In [16]:
import json # Used to read JSON files and convert them into Python objects (list/dict).
import os # Used to check whether a file already exists on our system.
import requests  # Used to send an HTTP request to download the file from the internet.


def download_and_load_file(file_path, url):
    if not os.path.exists(file_path):
        response = requests.get(url, timeout=30)
        response.raise_for_status()  # 404 error, server error, Network issue
        #Know more about 404 error: https://developer.mozilla.org/en-US/docs/Web/HTTP/Reference/Status/404
        
        text_data = response.text
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)

    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    return data

# params
file_path = "instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)

data = download_and_load_file(file_path, url)

print("Number of entries:", len(data))

Number of entries: 1100


In [19]:
print("Example entry:\n", data[2])

Example entry:
 {'instruction': 'Convert 45 kilometers to meters.', 'input': '', 'output': '45 kilometers is 45000 meters.'}


In [21]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text